# Results — measuring both dimensions, controls, and a real equivalence test

*Companion to Part II — Metabolic Scaling Theory and Biological Fractals.* CC-BY-4.0. This is a **reviewer's** notebook: it exists to test the paper's claims, not just to illustrate them.

Addresses MAJOR issues #4, #5, #6. Pulls the empirical images from CyVerse when available; otherwise uses synthetic objects of **known** dimension.

In [1]:
import sys, os
HERE = os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)
# fractal_review_utils.py lives next to these notebooks. Launch Jupyter from
# docs/notebooks/metabolic-scaling/review/ (or add that folder to sys.path).
import numpy as np
import matplotlib.pyplot as plt
import fractal_review_utils as u
print('utils loaded from', u.__file__)

utils loaded from /Users/tswetnam/github/fractal-notebooks/docs/notebooks/metabolic-scaling/review/fractal_review_utils.py


In [2]:
# --- Pull the author's empirical data from the CyVerse Data Store ---------
# Uses YOUR iRODS credentials. Authenticate first in a terminal:
#     gocmd init      # or:  iinit
# The collection is private; if you lack access the pull fails cleanly and we
# fall back to synthetic objects (and say so).
data_dir, status = u.cyverse_pull(u.DEFAULT_REMOTE, local='./data')
HAVE_DATA = data_dir is not None
print('CyVerse status :', status)
print('Local data dir :', data_dir)
print('HAVE_DATA      :', HAVE_DATA)

[cyverse_pull] No iRODS client found (looked for `gocmd` and `iget`). Install iCommands or gocmd and authenticate, then re-run.
CyVerse status : No iRODS client found (looked for `gocmd` and `iget`). Install iCommands or gocmd and authenticate, then re-run.
Local data dir : None
HAVE_DATA      : False


## 1. Range check: the named method (DBC) returns [2,3], not <2
The Methods call the technique *differential box-counting*. A true grayscale DBC lives in **[2,3]**; every value in Tables 1–4 is **<2**. Demonstrate the range on objects of known dimension:

In [3]:
# fBm surface (grayscale) -> DBC should be in [2,3]
surf = u.fbm_surface(256, H=0.6, seed=3)
dbc = u.differential_box_count(surf)
# Sierpinski carpet (binary) with an ALIGNED (base-3) ladder -> exact 1.893
carpet = u.sierpinski_carpet(order=6)
base3 = np.array([3**k for k in range(1,6)])
bbc_ok  = u.binary_box_count(carpet, sizes=base3)
bbc_bad = u.binary_box_count(carpet)  # mismatched power-of-2 ladder
print(f'grayscale DBC of fBm surface : D={dbc["dimension"]:.3f}  (range 2-3, as expected)')
print(f'binary carpet, aligned ladder: D={bbc_ok["dimension"]:.3f}  (true {u.SIERPINSKI_CARPET_DIM:.3f})')
print(f'binary carpet, WRONG ladder  : D={bbc_bad["dimension"]:.3f}  r2={bbc_bad["r2"]:.4f}  <-- high r2, WRONG D')

grayscale DBC of fBm surface : D=2.160  (range 2-3, as expected)
binary carpet, aligned ladder: D=1.893  (true 1.893)
binary carpet, WRONG ladder  : D=2.016  r2=0.9973  <-- high r2, WRONG D


**Note the last line:** `r² = 0.997` on a fit that returns the *wrong* dimension. The manuscript uses `μr² ≈ 0.99` as a quality signal (MINOR issue). High r² does not validate the dimension — report a CI on the slope instead.

## 2. The synthetic controls undercut the biological 'match' (MAJOR #6)
Table 1's synthetic objects have known/knowable dimensions; the tabulated method does not recover them (a space-filling Peano curve, true D=2, is reported at 1.85). Reproduce the compression toward ~1.5–1.6:

In [4]:
# The paper's own Table 1 synthetic values (mass dimension d_m):
paper_synth = {
  'Peano curve 1 (true D=2)': 1.846, 'Peano curve 2 (true D=2)': 1.803,
  'H-fractal': 1.760, 'Pythagoras tree 1': 1.607, 'Pythagoras tree 2': 1.655,
  "Barnsley's fern": 1.576, 'Fibonacci tree': 1.470,
}
print('Paper Table 1 synthetic mass dimensions (all < 2):')
for k,v in paper_synth.items(): print(f'  {k:<28} {v}')
vals = np.array(list(paper_synth.values()))
print(f'\nspread: {vals.min():.3f}-{vals.max():.3f}, mean {vals.mean():.3f}')
print('A space-filling Peano curve reading 1.8 (not 2.0) shows the estimator does')
print('not recover known dimensions; biological ~1.5 may be estimator central tendency.')

Paper Table 1 synthetic mass dimensions (all < 2):
  Peano curve 1 (true D=2)     1.846
  Peano curve 2 (true D=2)     1.803
  H-fractal                    1.76
  Pythagoras tree 1            1.607
  Pythagoras tree 2            1.655
  Barnsley's fern              1.576
  Fibonacci tree               1.47

spread: 1.470-1.846, mean 1.674
A space-filling Peano curve reading 1.8 (not 2.0) shows the estimator does
not recover known dimensions; biological ~1.5 may be estimator central tendency.


## 3. The equivalence test the Results assert but never run (MAJOR #4)
'Statistically indistinguishable from 3/2' is an **equivalence** claim. Use TOST against a pre-declared margin. We test the paper's own tabulated biological means against **both** candidate targets (3/2 and 4/3).

In [5]:
leaves   = [1.5384, 1.4844, 1.5525, 1.5135, 1.5083]      # Table 2
branches = [1.4946, 1.4775, 1.4549]                       # Table 3
canopy   = [1.3313, 1.5223, 1.4973, 1.4566, 1.5319, 1.5355]  # Table 4
for name, samp in [('leaves', leaves), ('branches', branches), ('canopy', canopy)]:
    for target in (1.5, 4/3):
        r = u.tost_equivalence(samp, target=target, margin=0.10)
        verdict = 'EQUIVALENT' if r['equivalent'] else 'not shown equivalent'
        ci = r['ci_90']
        print(f'{name:<9} vs {target:.3f} (+-0.10): {verdict:<20} '
              f'mean={r["mean"]:.3f} 90%CI=({ci[0]:.3f},{ci[1]:.3f}) n={r["n"]}')
    print()

leaves    vs 1.500 (+-0.10): EQUIVALENT           mean=1.519 90%CI=(1.494,1.545) n=5
leaves    vs 1.333 (+-0.10): not shown equivalent mean=1.519 90%CI=(1.494,1.545) n=5

branches  vs 1.500 (+-0.10): EQUIVALENT           mean=1.476 90%CI=(1.442,1.509) n=3
branches  vs 1.333 (+-0.10): not shown equivalent mean=1.476 90%CI=(1.442,1.509) n=3

canopy    vs 1.500 (+-0.10): EQUIVALENT           mean=1.479 90%CI=(1.415,1.543) n=6
canopy    vs 1.333 (+-0.10): not shown equivalent mean=1.479 90%CI=(1.415,1.543) n=6



**Reviewer reading:** at face value this is *good news for the authors* — with a ±0.10 margin all three biological samples are shown equivalent to **3/2** and **not** to 4/3, so the data, as tabulated, do favor 3/2 over 4/3. But three caveats keep this from being confirmation: (1) the margin ±0.10 is **arbitrary and undeclared** — and because its band width (0.20) exceeds the separation |3/2 − 4/3| ≈ 0.167, the two equivalence regions **overlap** in [1.40, 1.43], so the test is not guaranteed to discriminate for means that land there; (2) n = 3–6 is very low power; (3) the canopy case (90% CI 1.415–1.543, with the 1.33 rainforest outlier) only barely clears. Most importantly, celebrating 'data ≈ 3/2' is premature while the theory section still cannot decide **whether** the prediction is 3/2 or 4/3 (MAJOR #1). Fix the target first, then pre-register the margin.

## 4. Empirical run (only if CyVerse data are present)
If the pull succeeded, load the real images and compute **both** dimensions so the range issue is explicit on real data. Otherwise this cell is a no-op with a clear message (nothing fabricated).

In [6]:
import glob
if not HAVE_DATA:
    print('No CyVerse data (see status above). Skipping empirical run.')
    print('Authenticate with `gocmd init` and re-run the pull cell to enable this.')
else:
    try:
        from PIL import Image
    except ImportError:
        Image = None
        print('Pillow not installed; `pip install pillow` to load images.')
    imgs = []
    for ext in ('*.png','*.tif','*.tiff','*.jpg','*.jpeg'):
        imgs += glob.glob(os.path.join(data_dir, '**', ext), recursive=True)
    print(f'found {len(imgs)} image files under {data_dir}')
    if Image is not None:
        for path in sorted(imgs)[:10]:
            arr = np.asarray(Image.open(path).convert('L'), dtype=float)
            gray = u.differential_box_count(arr)['dimension']
            binary = u.binary_box_count(arr > arr.mean())['dimension']
            print(f'{os.path.basename(path):<32} DBC(2-3)={gray:.3f}  binary(1-2)={binary:.3f}')

No CyVerse data (see status above). Skipping empirical run.
Authenticate with `gocmd init` and re-run the pull cell to enable this.


### Reviewer note
Reporting *both* dimensions on every image resolves the ambiguity for readers: state which one is `d_M` in the tables and confirm it is in that estimator's valid range. Add per-image CIs and hold the box-size scaling range fixed across objects of different pixel counts (MINOR: resolution normalization).